In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

def setup_driver():
    """Set up Chrome driver with appropriate options"""
    chrome_options = Options()
    # chrome_options.add_argument("--headless")  # Remove this if you want to see the browser
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver

def scrape_tournament_metadata(url):
    """Scrape tournament metadata from gol.gg"""
    driver = setup_driver()
    
    try:
        print(f"Navigating to: {url}")
        driver.get(url)
        
        # Wait for the page to load
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        
        # Optional: Wait a bit more for dynamic content
        time.sleep(3)
        
        # Print the page HTML
        print("\n" + "="*50)
        print("PAGE HTML:")
        print("="*50)
        print(driver.page_source)
        
        # Also print the page title to verify we're on the right page
        print("\n" + "="*50)
        print("PAGE TITLE:")
        print("="*50)
        print(driver.title)
        
        return driver.page_source
        
    except Exception as e:
        print(f"Error occurred: {e}")
        return None
        
    finally:
        driver.quit()

def main():
    """Main function to run the scraper"""
    # Example URL - you can modify this
    url = "https://gol.gg/tournament/tournament-stats/LCK%20Spring%202022/"
    
    # Scrape the tournament page
    html_content = scrape_tournament_metadata(url)
    
    if html_content:
        print("\nSuccessfully retrieved HTML content!")
        # You can save to file if needed:
        with open("tournament_page.html", "w", encoding="utf-8") as f:
            f.write(html_content)
    else:
        print("Failed to retrieve HTML content!")

if __name__ == "__main__":
    main()

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import time

def scrape_tournament_metadata(url: str) -> pd.DataFrame:
    driver = webdriver.Chrome()
    driver.get(url)
    time.sleep(2)

    # az első "Tournament data" táblát fogjuk célozni
    table = driver.find_element(By.XPATH, "//caption[contains(., 'stats')]/ancestor::table")

    caption = table.find_element(By.TAG_NAME, "caption").text.replace(" stats", "")

    rows = table.find_elements(By.XPATH, ".//tbody/tr")

    data = {"tournament_name": caption}

    for row in rows:
        key = row.find_elements(By.TAG_NAME, "td")[0].text.strip(":")
        value = row.find_elements(By.TAG_NAME, "td")[1].text.strip()
        data[key] = value

    driver.quit()
    return pd.DataFrame([data])

df = scrape_tournament_metadata("https://gol.gg/tournament/tournament-stats/LCK%20Spring%202022/")
print(df)


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import time

BASE_URL = "https://gol.gg"

def scrape_tournament_matchlist(url: str) -> pd.DataFrame:
    driver = webdriver.Chrome()
    driver.get(url)
    time.sleep(2)

    table = driver.find_element(
        By.XPATH,
        "//h1[contains(., 'Match List')]/following::table[contains(@class, 'table_list')]"
    )

    rows = table.find_elements(By.XPATH, ".//tbody/tr")
    data = []

    for row in rows:
        cols = row.find_elements(By.TAG_NAME, "td")
        
        # üres vagy spacer row kihagyása
        if not cols:
            continue
        if len(cols) < 7:   # biztosítsuk hogy teljes sor
            continue
        # első cellában tényleg van-e link?
        links = cols[0].find_elements(By.TAG_NAME, "a")
        if not links:
            continue

        link_el = links[0]
        match_name = link_el.text.strip()

        relative_link = link_el.get_attribute("href")
        link = relative_link if relative_link.startswith("http") else BASE_URL + relative_link[2:]

        team1 = cols[1].text.strip()
        score = cols[2].text.strip()
        team2 = cols[3].text.strip()
        week = cols[4].text.strip()
        patch = cols[5].text.strip()
        date = cols[6].text.strip()

        data.append({
            "match_name": match_name,
            "team1": team1,
            "score": score,
            "team2": team2,
            "week": week,
            "patch": patch,
            "date": date,
            "link": link
        })

    driver.quit()
    return pd.DataFrame(data)

df_matches = scrape_tournament_matchlist(
    "https://gol.gg/tournament/tournament-matchlist/LCK%20Spring%202022/"
)

print(df_matches.head())


In [ ]:
# Score integers
df_matches['team1_score'] = df_matches['score'].str.split(" - ").str[0].astype(int)
df_matches['team2_score'] = df_matches['score'].str.split(" - ").str[1].astype(int)
df_matches['total_games'] = df_matches['team1_score'] + df_matches['team2_score']
df_matches['match_id'] = df_matches['link'].str.split("/").str[-3]

display(df_matches.head())

In [ ]:
# Game scraper

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time
import re
from typing import Dict, List, Any
import json

def setup_driver():
    """Setup Chrome driver with options"""
    chrome_options = Options()
    #chrome_options.add_argument('--headless')  # Háttérben fut
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver

def scrape_games(start_game_id: int, total_games: int, return_dataframe: bool = True):
    """
    Scrape League of Legends match data from gol.gg using Selenium
    
    Args:
        start_game_id: Starting game ID
        total_games: Number of games to scrape
        return_dataframe: If True, returns pandas DataFrame, else prints
    
    Returns:
        pandas.DataFrame if return_dataframe=True, else None
    """
    driver = setup_driver()
    
    all_rows = []
    
    try:
        for gid in range(start_game_id, start_game_id + total_games):
            print(f"\n{'='*80}")
            print(f"SCRAPING GAME ID: {gid}")
            print(f"{'='*80}\n")
            
            url = f"https://gol.gg/game/stats/{gid}/page-game/"
            
            try:
                driver.get(url)
                
                # Wait for page to load
                wait = WebDriverWait(driver, 10)
                wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
                
                # Additional wait for dynamic content
                time.sleep(2)
                
                # Get page source and parse
                html = driver.page_source
                soup = BeautifulSoup(html, 'html.parser')
                
                game_data = parse_game_data(driver, soup)
                
                if return_dataframe:
                    # Convert to row and append
                    row = flatten_game_data_to_row(gid, game_data)
                    row['match_id'] = start_game_id
                    all_rows.append(row)
                    print(f"✅ Game {gid} data collected")
                else:
                    # Print as before
                    print_game_data(game_data)
                
            except Exception as e:
                print(f"❌ Error scraping game {gid}: {e}")
                if return_dataframe:
                    # Add empty row with game_id
                    all_rows.append({'game_id': gid, 'error': str(e)})
                
            # Brief pause between requests
            time.sleep(1)
    
    finally:
        driver.quit()
    
    if return_dataframe and all_rows:
        df = pd.DataFrame(all_rows)
        # Reorder columns to put game_id first
        cols = ['match_id', 'game_id'] + [col for col in df.columns if col not in ['game_id', 'match_id']]
        df = df[cols]
        return df
    
    return None

def parse_game_data(driver, soup: BeautifulSoup) -> Dict[str, Any]:
    """Parse all game data from the page"""
    
    data = {
        'match_info': parse_match_info(soup),
        'blue_team': parse_team_data(soup, 'blue'),
        'red_team': parse_team_data(soup, 'red'),
        'timeline': parse_timeline(soup),
        'stats': parse_additional_stats(soup),
        'distributions': parse_distribution_stats(soup),
        'charts': parse_chart_data(driver, soup)
    }
    
    return data

def parse_match_info(soup: BeautifulSoup) -> Dict[str, Any]:
    """Parse basic match information"""
    info = {}
    
    # Match title
    h1 = soup.find('h1')
    if h1:
        info['match_title'] = h1.text.strip()
    
    # Tournament info
    tournament_link = soup.find('a', href=re.compile(r'tournament-stats'))
    if tournament_link:
        info['tournament'] = tournament_link.text.strip()
    
    # Date
    date_div = soup.find('div', class_='text-right', string=re.compile(r'\d{4}-\d{2}-\d{2}'))
    if date_div:
        info['date'] = date_div.text.strip()
    
    # Game time
    game_time_div = soup.find('div', class_='col-6 text-center')
    if game_time_div:
        time_h1 = game_time_div.find('h1')
        if time_h1:
            info['game_duration'] = time_h1.text.strip()
    
    # Patch version
    patch_div = soup.find('div', class_='col-3 text-right', string=re.compile(r'v\d+\.\d+'))
    if patch_div:
        info['patch'] = patch_div.text.strip()
    
    return info

def parse_team_data(soup: BeautifulSoup, side: str) -> Dict[str, Any]:
    """Parse team data (blue or red side)"""
    
    is_blue = side == 'blue'
    header_class = 'blue-line-header' if is_blue else 'red-line-header'
    line_class = 'blue_line' if is_blue else 'red_line'
    
    team_data = {
        'side': side.upper(),
        'result': None,
        'team_name': None,
        'stats': {},
        'objectives': {},
        'bans': [],
        'picks': [],
        'players': []
    }
    
    # Find team container
    team_container = soup.find('div', class_=header_class)
    if not team_container:
        return team_data
    
    # Team name and result
    team_link = team_container.find('a')
    if team_link:
        team_data['team_name'] = team_link.text.strip()
    
    result_text = team_container.text
    if 'WIN' in result_text:
        team_data['result'] = 'WIN'
    elif 'LOSS' in result_text:
        team_data['result'] = 'LOSS'
    
    # Get parent container for stats
    parent_col = team_container.find_parent('div', class_='col-12 col-sm-6')
    
    if parent_col:
        # Kills
        kills_span = parent_col.find('span', class_=f'score-box {line_class}')
        if kills_span:
            kills_text = kills_span.text.strip()
            try:
                team_data['stats']['kills'] = int(kills_text)
            except:
                pass
        
        # First blood
        first_blood_img = parent_col.find('img', alt='First Blood')
        team_data['objectives']['first_blood'] = first_blood_img is not None
        
        # Parse all score boxes
        score_boxes = parent_col.find_all('span', class_=f'score-box {line_class}')
        for box in score_boxes:
            img = box.find('img')
            if img:
                alt = img.get('alt', '')
                value = box.text.strip()
                
                if 'Kills' in alt:
                    try:
                        team_data['stats']['kills'] = int(value)
                    except:
                        pass
                elif 'Towers' in alt:
                    try:
                        team_data['stats']['towers'] = int(value)
                    except:
                        pass
                elif 'Dragons' in alt:
                    try:
                        team_data['stats']['dragons'] = int(value)
                    except:
                        pass
                elif 'Nashor' in alt or 'Barons' in alt:
                    try:
                        team_data['stats']['barons'] = int(value)
                    except:
                        pass
                elif 'Gold' in alt:
                    team_data['stats']['gold'] = value
        
        # First tower
        first_tower_img = parent_col.find('img', alt='First Tower')
        team_data['objectives']['first_tower'] = first_tower_img is not None
        
        # Dragon types
        dragon_types = []
        dragon_imgs = parent_col.find_all('img', src=re.compile(r'dragon'))
        for img in dragon_imgs:
            src = img.get('src', '')
            if 'cloud-dragon' in src:
                dragon_types.append('Cloud')
            elif 'hextech-dragon' in src:
                dragon_types.append('Hextech')
            elif 'mountain-dragon' in src:
                dragon_types.append('Mountain')
            elif 'infernal-dragon' in src:
                dragon_types.append('Infernal')
            elif 'ocean-dragon' in src:
                dragon_types.append('Ocean')
            elif 'chemtech-dragon' in src:
                dragon_types.append('Chemtech')
        
        team_data['objectives']['dragon_types'] = dragon_types
        
        # Bans
        ban_row = parent_col.find('div', string='Bans')
        if ban_row:
            ban_container = ban_row.find_next_sibling('div')
            if ban_container:
                ban_imgs = ban_container.find_all('img', class_='champion_icon_medium')
                team_data['bans'] = [img.get('alt', 'Unknown') for img in ban_imgs]
        
        # Picks
        pick_row = parent_col.find('div', string='Picks')
        if pick_row:
            pick_container = pick_row.find_next_sibling('div')
            if pick_container:
                pick_imgs = pick_container.find_all('img', class_='champion_icon_medium')
                team_data['picks'] = [img.get('alt', 'Unknown') for img in pick_imgs]
    
    # Players
    all_tables = soup.find_all('table', class_='playersInfosLine')
    for table in all_tables:
        # Check if this table belongs to the current team
        table_header = table.find('tr', class_=header_class)
        if table_header:
            rows = table.find('tbody').find_all('tr') if table.find('tbody') else []
            for row in rows:
                player = parse_player_row(row)
                if player:
                    team_data['players'].append(player)
            break
    
    return team_data

def parse_player_row(row) -> Dict[str, Any]:
    """Parse individual player data from table row"""
    
    player = {}
    
    # Champion
    champion_img = row.find('img', class_='champion_icon')
    if champion_img:
        player['champion'] = champion_img.get('alt', 'Unknown')
    
    # Player name
    player_link = row.find('a', class_='link-blanc')
    if player_link:
        player['player_name'] = player_link.text.strip()
    
    # ADJUNK HOZZÁ EGY ELLENŐRZÉST:
    # Ha nincs player_name ÉS nincs champion, akkor ez egy rune sor, kihagyjuk
    if not player.get('player_name') and not player.get('champion'):
        return None
    
    # Ha van champion de nincs player_name, akkor is kihagyjuk (tooltip rész)
    if player.get('champion') and not player.get('player_name'):
        return None
    
    # Get all cells
    cells = row.find_all('td')
    
    # KDA (usually in a cell with text-align:center containing '/')
    for cell in cells:
        text = cell.text.strip()
        if '/' in text and re.match(r'\d+/\d+/\d+', text):
            player['kda'] = text
            break
    
    # CS (numeric value in appropriate cell)
    cs_cells = row.find_all('td', style=re.compile(r'text-align:center'))
    for cell in cs_cells:
        text = cell.text.strip()
        if text.isdigit() and int(text) > 20:  # CS is usually > 20
            player['cs'] = int(text)
            break
    
    # Summoner spells
    summoner_imgs = row.find_all('img', alt='Summoner spell')
    if summoner_imgs:
        player['summoner_spells'] = []
        for img in summoner_imgs[:2]:
            src = img.get('src', '')
            spell_name = src.split('/')[-1].replace('.png', '').replace('Summoner', '')
            player['summoner_spells'].append(spell_name)
    
    # Items
    item_imgs = row.find_all('img', alt='Items')
    if item_imgs:
        player['items'] = []
        for img in item_imgs:
            src = img.get('src', '')
            item_id = src.split('/')[-1].replace('.png', '')
            player['items'].append(item_id)
    
    # Keystone rune
    rune_imgs = row.find_all('img', alt='Runes')
    if rune_imgs:
        first_rune = rune_imgs[0]
        src = first_rune.get('src', '')
        rune_id = src.split('/')[-1].replace('.png', '')
        player['keystone'] = rune_id
    
    return player if player else None

def parse_timeline(soup: BeautifulSoup) -> List[Dict[str, Any]]:
    """Parse timeline events"""
    
    timeline = []
    
    timeline_div = soup.find('div', class_='col-12 flex-wrap')
    if timeline_div:
        event_spans = timeline_div.find_all('span', style='display:inline-block')
        
        for span in event_spans:
            blue_action = span.find('span', class_='blue_action')
            red_action = span.find('span', class_='red_action')
            
            action = blue_action or red_action
            if action:
                side = 'BLUE' if blue_action else 'RED'
                
                # Get time from text
                text = action.get_text(strip=True)
                
                # Get event type from image
                img = action.find('img')
                event_type = 'Unknown'
                if img:
                    alt = img.get('alt', '')
                    src = img.get('src', '')
                    
                    if alt:
                        event_type = alt
                    elif 'dragon' in src:
                        if 'cloud' in src:
                            event_type = 'Cloud Drake'
                        elif 'hextech' in src:
                            event_type = 'Hextech Drake'
                        elif 'mountain' in src:
                            event_type = 'Mountain Drake'
                        elif 'infernal' in src:
                            event_type = 'Infernal Drake'
                        elif 'ocean' in src:
                            event_type = 'Ocean Drake'
                    elif 'herald' in src:
                        event_type = 'Rift Herald'
                    elif 'baron' in src or 'nashor' in src:
                        event_type = 'Baron Nashor'
                    elif 'tower' in src:
                        event_type = 'Tower'
                    elif 'firstblood' in src:
                        event_type = 'First Blood'
                
                timeline.append({
                    'side': side,
                    'time': text,
                    'type': event_type
                })
    
    return timeline

def parse_distribution_stats(soup: BeautifulSoup) -> Dict[str, Any]:
    """Parse gold and damage distribution tables"""
    
    distributions = {}
    
    # Gold Distribution
    gold_header = soup.find('th', string='Gold distribution')
    if gold_header:
        table = gold_header.find_parent('table')
        if table:
            # Find the small_table with percentages
            small_table = table.find('table', class_='small_table')
            if small_table:
                gold_dist = {}
                rows = small_table.find_all('tr')[1:]  # Skip header
                for row in rows:
                    cols = row.find_all('td')
                    if len(cols) == 3:
                        role = cols[0].text.strip()
                        blue_pct = cols[1].text.strip()
                        red_pct = cols[2].text.strip()
                        gold_dist[role] = {'blue': blue_pct, 'red': red_pct}
                distributions['gold_distribution'] = gold_dist
    
    # Damage Distribution
    dmg_header = soup.find('th', string='Damage distribution')
    if dmg_header:
        table = dmg_header.find_parent('table')
        if table:
            small_table = table.find('table', class_='small_table')
            if small_table:
                dmg_dist = {}
                rows = small_table.find_all('tr')[1:]  # Skip header
                for row in rows:
                    cols = row.find_all('td')
                    if len(cols) == 3:
                        role = cols[0].text.strip()
                        blue_cell = cols[1]
                        red_cell = cols[2]
                        
                        # Extract percentage (might have tooltip with DPM)
                        blue_span = blue_cell.find('span')
                        red_span = red_cell.find('span')
                        
                        blue_pct = blue_span.text.strip() if blue_span else blue_cell.text.strip()
                        red_pct = red_span.text.strip() if red_span else red_cell.text.strip()
                        
                        # Extract DPM from tooltip if exists
                        blue_dpm = blue_span.get('data-original-title', '') if blue_span else ''
                        red_dpm = red_span.get('data-original-title', '') if red_span else ''
                        
                        dmg_dist[role] = {
                            'blue': {'percentage': blue_pct, 'dpm': blue_dpm},
                            'red': {'percentage': red_pct, 'dpm': red_dpm}
                        }
                distributions['damage_distribution'] = dmg_dist
    
    return distributions

def parse_additional_stats(soup: BeautifulSoup) -> Dict[str, Any]:
    """Parse additional statistics"""
    
    stats = {}
    
    # Plates
    plates_header = soup.find('th', string='Plates')
    if plates_header:
        table = plates_header.find_parent('table')
        if table:
            rows = table.find_all('div', class_='row pb-3')
            plates_data = {}
            
            for row in rows:
                cols = row.find_all('div', class_=re.compile(r'col-\d+'))
                if len(cols) >= 3:
                    label = cols[0].text.strip()
                    blue_val = cols[1].text.strip()
                    red_val = cols[2].text.strip()
                    
                    if label and blue_val and red_val and label != 'Plates':
                        plates_data[label] = {'blue': blue_val, 'red': red_val}
            
            if plates_data:
                stats['plates'] = plates_data
    
    return stats

def parse_chart_data(driver, soup: BeautifulSoup) -> Dict[str, Any]:
    """Parse chart data from Chart.js instances"""
    
    chart_data = {}
    
    try:
        # Extract Gold Timeline from Chart.js
        gold_timeline = driver.execute_script("""
            var canvas = document.getElementById('GoldLine');
            if (typeof Chart !== 'undefined' && Chart.instances) {
                var instances = Chart.instances;
                for (var id in instances) {
                    if (instances[id].canvas && instances[id].canvas.id === 'GoldLine') {
                        var datasets = instances[id].data.datasets;
                        var goldDataset = null;
                        
                        // Find the dataset with label 'Gold'
                        for (var i = 0; i < datasets.length; i++) {
                            if (datasets[i].label === 'Gold' || datasets[i].data.some(v => v > 0)) {
                                goldDataset = datasets[i];
                                break;
                            }
                        }
                        
                        if (goldDataset) {
                            return {
                                labels: instances[id].data.labels,
                                data: goldDataset.data
                            };
                        }
                    }
                }
            }
            return null;
        """)
        
        if gold_timeline:
            chart_data['gold_timeline'] = {
                'time': gold_timeline['labels'],
                'gold_diff': gold_timeline['data']
            }
    
    except Exception as e:
        print(f"  ⚠️  Could not extract gold timeline: {e}")
    
    # Vision Chart
    try:
        vision_data = driver.execute_script("""
            var canvas = document.getElementById('eVisionChart');
            if (typeof Chart !== 'undefined' && Chart.instances) {
                var instances = Chart.instances;
                for (var id in instances) {
                    if (instances[id].canvas && instances[id].canvas.id === 'eVisionChart') {
                        var datasets = instances[id].data.datasets;
                        return {
                            labels: instances[id].data.labels,
                            datasets: datasets.map(d => ({
                                label: d.label,
                                data: d.data
                            }))
                        };
                    }
                }
            }
            return null;
        """)
        
        if vision_data and len(vision_data['datasets']) >= 2:
            # Datasets: [DRX (red), T1 (blue)] or vice versa
            ds1 = vision_data['datasets'][0]
            ds2 = vision_data['datasets'][1]
            
            # Figure out which is which (usually DRX is first, T1 is second)
            chart_data['vision'] = {
                'labels': vision_data['labels'],
                'team1': {'name': ds1['label'], 'data': ds1['data']},
                'team2': {'name': ds2['label'], 'data': ds2['data']}
            }
    
    except Exception as e:
        print(f"  ⚠️  Could not extract vision data: {e}")
    
    # Jungle Share Chart
    try:
        jungle_data = driver.execute_script("""
            var canvas = document.getElementById('counterChart');
            if (typeof Chart !== 'undefined' && Chart.instances) {
                var instances = Chart.instances;
                for (var id in instances) {
                    if (instances[id].canvas && instances[id].canvas.id === 'counterChart') {
                        var datasets = instances[id].data.datasets;
                        return {
                            labels: instances[id].data.labels,
                            datasets: datasets.map(d => ({
                                data: d.data
                            }))
                        };
                    }
                }
            }
            return null;
        """)
        
        if jungle_data and len(jungle_data['datasets']) >= 2:
            blue_data = jungle_data['datasets'][0]['data']
            red_data = jungle_data['datasets'][1]['data']
            
            chart_data['jungle_share'] = {
                'at_15min': {'blue': blue_data[0], 'red': red_data[0]},
                'end_game': {'blue': blue_data[1], 'red': red_data[1]}
            }
    
    except Exception as e:
        print(f"  ⚠️  Could not extract jungle share: {e}")
    
    return chart_data

def flatten_game_data_to_row(game_id: int, data: Dict[str, Any]) -> Dict[str, Any]:
    """Flatten nested game data into a single row for DataFrame"""
    
    row = {
        'game_id': game_id,
    }
    
    # Match info
    match_info = data.get('match_info', {})
    row['match_title'] = match_info.get('match_title', '')
    row['tournament'] = match_info.get('tournament', '')
    row['date'] = match_info.get('date', '')
    row['game_duration'] = match_info.get('game_duration', '')
    row['patch'] = match_info.get('patch', '')
    
    # Team data - Blue side
    blue = data.get('blue_team', {})
    row['blue_team'] = blue.get('team_name', '')
    row['blue_result'] = blue.get('result', '')
    row['blue_kills'] = blue.get('stats', {}).get('kills', 0)
    row['blue_towers'] = blue.get('stats', {}).get('towers', 0)
    row['blue_dragons'] = blue.get('stats', {}).get('dragons', 0)
    row['blue_barons'] = blue.get('stats', {}).get('barons', 0)
    row['blue_gold'] = blue.get('stats', {}).get('gold', '')
    row['blue_first_blood'] = blue.get('objectives', {}).get('first_blood', False)
    row['blue_first_tower'] = blue.get('objectives', {}).get('first_tower', False)
    row['blue_dragon_types'] = ', '.join(blue.get('objectives', {}).get('dragon_types', []))
    row['blue_bans'] = ', '.join(blue.get('bans', []))
    row['blue_picks'] = ', '.join(blue.get('picks', []))
    
    # Blue players
    for i, player in enumerate(blue.get('players', []), 1):
        prefix = f'blue_p{i}'
        row[f'{prefix}_name'] = player.get('player_name', '')
        row[f'{prefix}_champion'] = player.get('champion', '')
        row[f'{prefix}_kda'] = player.get('kda', '')
        row[f'{prefix}_cs'] = player.get('cs', 0)
        row[f'{prefix}_summoners'] = ', '.join(player.get('summoner_spells', []))
        row[f'{prefix}_keystone'] = player.get('keystone', '')
    
    # Team data - Red side
    red = data.get('red_team', {})
    row['red_team'] = red.get('team_name', '')
    row['red_result'] = red.get('result', '')
    row['red_kills'] = red.get('stats', {}).get('kills', 0)
    row['red_towers'] = red.get('stats', {}).get('towers', 0)
    row['red_dragons'] = red.get('stats', {}).get('dragons', 0)
    row['red_barons'] = red.get('stats', {}).get('barons', 0)
    row['red_gold'] = red.get('stats', {}).get('gold', '')
    row['red_first_blood'] = red.get('objectives', {}).get('first_blood', False)
    row['red_first_tower'] = red.get('objectives', {}).get('first_tower', False)
    row['red_dragon_types'] = ', '.join(red.get('objectives', {}).get('dragon_types', []))
    row['red_bans'] = ', '.join(red.get('bans', []))
    row['red_picks'] = ', '.join(red.get('picks', []))
    
    # Red players
    for i, player in enumerate(red.get('players', []), 1):
        prefix = f'red_p{i}'
        row[f'{prefix}_name'] = player.get('player_name', '')
        row[f'{prefix}_champion'] = player.get('champion', '')
        row[f'{prefix}_kda'] = player.get('kda', '')
        row[f'{prefix}_cs'] = player.get('cs', 0)
        row[f'{prefix}_summoners'] = ', '.join(player.get('summoner_spells', []))
        row[f'{prefix}_keystone'] = player.get('keystone', '')
    
    # Timeline events (as JSON string or count)
    timeline = data.get('timeline', [])
    row['timeline_events'] = len(timeline)
    row['timeline_json'] = json.dumps(timeline)
    
    # Stats
    stats = data.get('stats', {})
    plates = stats.get('plates', {})
    for plate_type, values in plates.items():
        safe_name = plate_type.replace(' ', '_').lower()
        row[f'{safe_name}_blue'] = values.get('blue', '')
        row[f'{safe_name}_red'] = values.get('red', '')
    
    # Distributions
    distributions = data.get('distributions', {})
    
    # Gold distribution
    gold_dist = distributions.get('gold_distribution', {})
    for role, values in gold_dist.items():
        row[f'gold_dist_{role.lower()}_blue'] = values.get('blue', '')
        row[f'gold_dist_{role.lower()}_red'] = values.get('red', '')
    
    # Damage distribution
    dmg_dist = distributions.get('damage_distribution', {})
    for role, values in dmg_dist.items():
        blue_info = values.get('blue', {})
        red_info = values.get('red', {})
        row[f'dmg_dist_{role.lower()}_blue_pct'] = blue_info.get('percentage', '')
        row[f'dmg_dist_{role.lower()}_blue_dpm'] = blue_info.get('dpm', '')
        row[f'dmg_dist_{role.lower()}_red_pct'] = red_info.get('percentage', '')
        row[f'dmg_dist_{role.lower()}_red_dpm'] = red_info.get('dpm', '')
    
    # Charts
    charts = data.get('charts', {})
    
    # Gold timeline - store as JSON or key metrics
    gold_timeline = charts.get('gold_timeline', {})
    if gold_timeline:
        row['gold_timeline_json'] = json.dumps(gold_timeline)
        # Also store some key points
        gold_data = gold_timeline.get('gold_diff', [])
        if len(gold_data) > 0:
            row['gold_diff_start'] = gold_data[0] if len(gold_data) > 0 else 0
            row['gold_diff_10min'] = gold_data[10] if len(gold_data) > 10 else 0
            row['gold_diff_15min'] = gold_data[15] if len(gold_data) > 15 else 0
            row['gold_diff_20min'] = gold_data[20] if len(gold_data) > 20 else 0
            row['gold_diff_end'] = gold_data[-1] if gold_data else 0
    
    # Vision
    vision = charts.get('vision', {})
    if vision:
        team1 = vision.get('team1', {})
        team2 = vision.get('team2', {})
        if team1.get('data'):
            row['vision_team1_wards_destroyed'] = team1['data'][0]
            row['vision_team1_wards_placed'] = team1['data'][1]
        if team2.get('data'):
            row['vision_team2_wards_destroyed'] = team2['data'][0]
            row['vision_team2_wards_placed'] = team2['data'][1]
    
    # Jungle share
    jungle = charts.get('jungle_share', {})
    if jungle:
        row['jungle_share_15min_blue'] = jungle.get('at_15min', {}).get('blue', 0)
        row['jungle_share_15min_red'] = jungle.get('at_15min', {}).get('red', 0)
        row['jungle_share_end_blue'] = jungle.get('end_game', {}).get('blue', 0)
        row['jungle_share_end_red'] = jungle.get('end_game', {}).get('red', 0)
    
    return row

def print_game_data(data: Dict[str, Any]):
    """Print game data in a structured format"""
    
    # Match info
    print("📊 MATCH INFORMATION")
    print("─" * 80)
    if data['match_info']:
        for key, value in data['match_info'].items():
            print(f"  {key.replace('_', ' ').title()}: {value}")
    else:
        print("  No match information found")
    
    # Teams
    for team_key in ['blue_team', 'red_team']:
        team = data[team_key]
        icon = '🔵' if team_key == 'blue_team' else '🔴'
        
        print(f"\n{icon} {team['team_name'] or 'Unknown Team'} - {team['result'] or 'N/A'}")
        print("─" * 80)
        
        # Team stats
        if team['stats']:
            print(f"  Stats:")
            for stat, value in team['stats'].items():
                print(f"    • {stat.capitalize()}: {value}")
        
        # Objectives
        if team['objectives']:
            print(f"  Objectives:")
            for obj, value in team['objectives'].items():
                if isinstance(value, list):
                    if value:
                        print(f"    • {obj.replace('_', ' ').title()}: {', '.join(value)}")
                else:
                    print(f"    • {obj.replace('_', ' ').title()}: {'✓' if value else '✗'}")
        
        # Bans and Picks
        if team['bans']:
            print(f"  Bans: {', '.join(team['bans'])}")
        if team['picks']:
            print(f"  Picks: {', '.join(team['picks'])}")
        
        # Players
        if team['players']:
            print(f"  Players:")
            for i, player in enumerate(team['players'], 1):
                print(f"    {i}. {player.get('player_name', 'Unknown')} ({player.get('champion', 'Unknown')})")
                print(f"       KDA: {player.get('kda', 'N/A')} | CS: {player.get('cs', 'N/A')}")
                if player.get('summoner_spells'):
                    print(f"       Summoners: {', '.join(player['summoner_spells'])}")
                if player.get('keystone'):
                    print(f"       Keystone: {player['keystone']}")
    
    # Timeline
    if data['timeline']:
        print(f"\n⏱️  TIMELINE")
        print("─" * 80)
        for event in data['timeline']:
            side_icon = '🔵' if event['side'] == 'BLUE' else '🔴'
            print(f"  {event['time']} - {side_icon} {event['type']}")
    
    # Additional stats
    if data['stats']:
        print(f"\n📈 ADDITIONAL STATISTICS")
        print("─" * 80)
        print(json.dumps(data['stats'], indent=2))

    if data.get('distributions'):
        print(f"\n💰 GOLD & DAMAGE DISTRIBUTION")
        print("─" * 80)
        
        if 'gold_distribution' in data['distributions']:
            print("\n  Gold Distribution:")
            for role, values in data['distributions']['gold_distribution'].items():
                print(f"    {role}: Blue {values['blue']} | Red {values['red']}")
        
        if 'damage_distribution' in data['distributions']:
            print("\n  Damage Distribution:")
            for role, values in data['distributions']['damage_distribution'].items():
                blue_info = values['blue']
                red_info = values['red']
                blue_str = f"{blue_info['percentage']}"
                red_str = f"{red_info['percentage']}"
                if blue_info.get('dpm'):
                    blue_str += f" ({blue_info['dpm']})"
                if red_info.get('dpm'):
                    red_str += f" ({red_info['dpm']})"
                print(f"    {role}: Blue {blue_str} | Red {red_str}")

    # Charts
    if data.get('charts'):
        print(f"\n📊 CHART DATA")
        print("─" * 80)
        
        if 'gold_timeline' in data['charts']:
            gold_tl = data['charts']['gold_timeline']
            print(f"\n  Gold Timeline ({len(gold_tl['gold_diff'])} data points):")
            # Show first, some middle points, and last
            indices = [0, 5, 10, 15, 20, -1]
            for i in indices:
                if i < len(gold_tl['time']):
                    time_val = gold_tl['time'][i]
                    gold_val = gold_tl['gold_diff'][i]
                    print(f"    {time_val:>3} min: {gold_val:>6} gold advantage")
        
        if 'vision' in data['charts']:
            vision = data['charts']['vision']
            print(f"\n  Vision Stats:")
            print(f"    Labels: {', '.join(vision['labels'])}")
            print(f"    {vision['team1']['name']}: {vision['team1']['data']}")
            print(f"    {vision['team2']['name']}: {vision['team2']['data']}")
        
        if 'jungle_share' in data['charts']:
            jungle = data['charts']['jungle_share']
            print(f"\n  Jungle Share:")
            print(f"    At 15 min - Blue: {jungle['at_15min']['blue']:.1f}%, Red: {jungle['at_15min']['red']:.1f}%")
            print(f"    End game - Blue: {jungle['end_game']['blue']:.1f}%, Red: {jungle['end_game']['red']:.1f}%")

In [ ]:
# Usage: Game scraper

df = scrape_games(38812, 2, return_dataframe=True)

for col in df.columns:
    print(f"{col}: {df[col][0]}")